In [ ]:
#Setup & Imports
import os
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from collections import Counter
import random


#Set device for the notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
#Global ELIZA Scripts including English and Burmese
SCRIPTS = {
    "en": {
        "initials": ["How do you do. Please tell me your problem.", "Is something troubling you?"],
        "finals": ["Goodbye. It was nice talking to you."],
        "quits": ["bye", "quit", "exit"],
        "pres": {"don't": "dont", "i'm": "i am", "recollect": "remember", "machine": "computer"},
        "posts": {"am": "are", "i": "you", "my": "your", "me": "you", "your": "my"},
        "keywords": [
            [r'(.*) die (.*)', ["Please don't talk like that. Tell me more about your feelings."], 10],
            [r'(.*) problem (.*)', ["Tell me more about this problem.", "How does it make you feel?"], 8],
            [r'(.*)', ["Please tell me more.", "I see.", "Can you elaborate?"], 0]
        ]
    },
    "my": {
        "initials": ["မင်္ဂလာပါ။ ဘာတွေအခက်အခဲရှိလဲ ပြောပြပါလား။", "ဘာတွေများ အကူအညီလိုနေသလဲ။"],
        "finals": ["ကောင်းပါပြီ။ နောက်မှပြန်တွေ့ကြတာပေါ့။", "နှုတ်ဆက်ပါတယ်။"],
        "quits": ["bye", "ထွက်မည်", "သွားတော့မယ်", "quit"],

        "pres": {"ကျွန်တော်": "ကျနော်", "ကျွန်မ": "ကျမ"},

        "posts": {"ကျွန်တော်": "ခင်ဗျား", "ကျနော်": "မင်း", "ကျွန်တော့်": "ခင်ဗျားရဲ့"},
        "keywords": [

            [r'(.*)သေ(.*)', ["ဒီလိုမပြောပါနဲ့။ ဘယ်လိုခံစားနေရလဲ ထပ်ပြောပြပေးပါ။"], 10],
            [r'(.*)ပြဿနာ(.*)', ["ဒီပြဿနာအကြောင်း ထပ်ပြောပြပါလား။", "ဒါက သင့်ကို ဘယ်လိုခံစားရစေလဲ။"], 8],
            [r'(.*)ဝမ်းနည်း(.*)', ["ဘာကြောင့် ဝမ်းနည်းနေရတာလဲ။", "ဘယ်လောက်ကြာပြီလဲ။"], 5],
            [r'(.*)', ["ဆက်ပြောပါဦး။", "ဟုတ်ကဲ့၊ နားလည်ပါတယ်။", "နောက်ထပ် ရှင်းပြပေးနိုင်မလား။"], 0]
        ]
    }
}

In [ ]:
#PyTorch Datasets, Models, and Sylbreak Segmenter
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset

def segment_myanmar(text):
    text = str(text)
    # The standard sylbreak regex pattern for Myanmar syllables
    sylbreak_regex = re.compile(r"(?<!္)([က-ဪဿ၊-၏]|[၀-၉]+|[^က-၏]+)(?![ှျ]?[့္်])")

    # Insert spaces between syllables
    segmented = sylbreak_regex.sub(r" \1", text)

    # Clean out generic punctuation but keep Myanmar unicode
    segmented = re.sub(r'[^\w\s\u1000-\u109F]', '', segmented.lower())

    # Collapse multiple spaces into one
    return re.sub(r'\s+', ' ', segmented).strip()

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, word2id, max_len=50):
        self.texts = texts
        self.labels = labels
        self.word2id = word2id
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        # Apply the advanced segmenter
        clean_text = segment_myanmar(self.texts[idx])
        seq = [self.word2id.get(w, 1) for w in clean_text.split()][:self.max_len]
        padding = [0] * (self.max_len - len(seq))
        return torch.tensor(seq + padding), torch.tensor(self.labels[idx])

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(x * weights, dim=1), weights

class EmotionalBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        context, weights = self.attention(lstm_out)
        return self.fc(context), weights

In [ ]:


class HybridEliza:
    def __init__(self, lang="my", model_path="eliza_eq_my.pth"):
        self.lang = lang
        self.script = SCRIPTS[lang]
        self.script["keywords"].sort(key=lambda x: x[2], reverse=True)

        self.model_path = model_path
        self.device = device
        self.word2id = {"<PAD>": 0, "<UNK>": 1}

        # Verify these match the integer labels in your CSV!
        self.id2label = {0: "Sadness", 1: "Joy", 2: "Love", 3: "Anger", 4: "Fear", 5: "Surprise"}
        self.num_classes = len(self.id2label)
        self.model = None

    def build_vocab(self, texts):
        # Apply the segmenter to build a proper syllable vocabulary
        clean_texts = [segment_myanmar(t) for t in texts]
        words = Counter([w for t in clean_texts for w in t.split()])
        for i, (w, _) in enumerate(words.most_common(5000), 2):
            self.word2id[w] = i

    def train(self, data_path, epochs=15, lr=0.001, batch_size=32, val_split=0.1):
        df = pd.read_excel(data_path)

        # Safety check for label count mismatch
        unique_labels = df['Label'].nunique()
        if unique_labels != self.num_classes:
            print(f"[!] Warning: Dataset has {unique_labels} classes, but model expects {self.num_classes}.")

        self.build_vocab(df['Text'])

        full_dataset = EmotionDataset(df['Text'].tolist(), df['Label'].tolist(), self.word2id)
        val_size = int(len(full_dataset) * val_split)
        train_size = len(full_dataset) - val_size
        train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size)

        self.model = EmotionalBiLSTM(len(self.word2id), 128, 64, self.num_classes).to(self.device)
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        print(f"[*] Training on {self.device} ({self.num_classes} Classes)...")
        for epoch in range(epochs):
            self.model.train()
            total_loss = 0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                outputs, _ = self.model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            val_acc = self.evaluate(val_loader)
            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2%}")

        torch.save({'state': self.model.state_dict(), 'vocab': self.word2id}, self.model_path)
        print("[*] Model saved successfully.")

    def evaluate(self, loader):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                outputs, _ = self.model(x)
                _, predicted = torch.max(outputs.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()
        return correct / total

    def load_model(self):
        if os.path.exists(self.model_path):
            checkpoint = torch.load(self.model_path, map_location=self.device)
            self.word2id = checkpoint['vocab']
            self.model = EmotionalBiLSTM(len(self.word2id), 128, 64, self.num_classes).to(self.device)
            self.model.load_state_dict(checkpoint['state'])
            self.model.eval()
            print("[*] Model loaded successfully.")
        else:
            print("[!] Model file not found. Please train first.")

    def get_eq(self, text):
        if not self.model: return "Neutral", 0.0

        # Apply the segmenter to user input during chat
        clean_text = segment_myanmar(text)
        tokens = [self.word2id.get(w, 1) for w in clean_text.split()][:50]
        tokens += [0] * (50 - len(tokens))

        with torch.no_grad():
            tensor_input = torch.tensor([tokens]).to(self.device)
            output, weights = self.model(tensor_input)
            probs = torch.softmax(output, dim=1)
            idx = torch.argmax(probs).item()
            return self.id2label[idx], probs[0][idx].item()

    def rule_respond(self, text):
        text = text.lower()
        for k, v in self.script["pres"].items(): text = text.replace(k, v)
        for pattern, resps, rank in self.script["keywords"]:
            match = re.search(pattern, text)
            if match:
                resp = random.choice(resps)
                frags = [self.reflect(g) for g in match.groups() if g]
                try:
                    return resp.format(*frags) if frags else resp
                except IndexError:
                    return resp
        return random.choice(self.script["keywords"][-1][1])

    def reflect(self, fragment):
        return " ".join([self.script["posts"].get(w, w) for w in fragment.split()])

In [ ]:
#Initialize and Train Model
eliza_my = HybridEliza(lang="my", model_path="eliza_eq_my.pth")

eliza_my.train(data_path="emotion_myanmar_dataset_V8.xlsx", epochs=30, batch_size=32)

[*] Training on cpu (6 Classes)...
Epoch 1/30 | Loss: 1.7709 | Val Acc: 36.69%
Epoch 2/30 | Loss: 1.5070 | Val Acc: 55.40%
Epoch 3/30 | Loss: 0.9698 | Val Acc: 63.31%
Epoch 4/30 | Loss: 0.6713 | Val Acc: 61.87%
Epoch 5/30 | Loss: 0.4670 | Val Acc: 64.75%
Epoch 6/30 | Loss: 0.3457 | Val Acc: 61.15%
Epoch 7/30 | Loss: 0.2294 | Val Acc: 65.47%
Epoch 8/30 | Loss: 0.1554 | Val Acc: 66.19%
Epoch 9/30 | Loss: 0.1026 | Val Acc: 65.47%
Epoch 10/30 | Loss: 0.0723 | Val Acc: 66.91%
Epoch 11/30 | Loss: 0.0512 | Val Acc: 67.63%
Epoch 12/30 | Loss: 0.0377 | Val Acc: 68.35%
Epoch 13/30 | Loss: 0.0275 | Val Acc: 67.63%
Epoch 14/30 | Loss: 0.0214 | Val Acc: 68.35%
Epoch 15/30 | Loss: 0.0168 | Val Acc: 67.63%
Epoch 16/30 | Loss: 0.0130 | Val Acc: 67.63%
Epoch 17/30 | Loss: 0.0117 | Val Acc: 67.63%
Epoch 18/30 | Loss: 0.0101 | Val Acc: 67.63%
Epoch 19/30 | Loss: 0.0089 | Val Acc: 69.06%
Epoch 20/30 | Loss: 0.0077 | Val Acc: 66.91%
Epoch 21/30 | Loss: 0.0133 | Val Acc: 67.63%
Epoch 22/30 | Loss: 0.0248 | 

In [ ]:
eliza_my.load_model()
print(f"ELIZA: {random.choice(SCRIPTS[eliza_my.lang]['initials'])}")

while True:
    user_in = input("You: ")
    if user_in.lower() in SCRIPTS[eliza_my.lang]["quits"]:
        print(f"ELIZA: {random.choice(SCRIPTS[eliza_my.lang]['finals'])}")
        break

    resp = eliza_my.rule_respond(user_in)
    emotion, score = eliza_my.get_eq(user_in)

    print(f"ELIZA: {resp}")
    print(f"[EQ Analysis]: Predicted Emotion: {emotion} ({score:.2%})\n")

[*] Model loaded successfully.
ELIZA: မင်္ဂလာပါ။ ဘာတွေအခက်အခဲရှိလဲ ပြောပြပါလား။
You: ချစ်စရာလေးဗျာ
ELIZA: ဟုတ်ကဲ့၊ နားလည်ပါတယ်။
[EQ Analysis]: Predicted Emotion: Love (99.89%)

You: ဟာ ကြောက်စရာကောင်းလိုက်တာဗျာ
ELIZA: ဆက်ပြောပါဦး။
[EQ Analysis]: Predicted Emotion: Fear (99.95%)

You: quit
ELIZA: ကောင်းပါပြီ။ နောက်မှပြန်တွေ့ကြတာပေါ့။


To add the evaluation metrics